# Drug-Surfactant API-Based Workflow Demo

This notebook demonstrates the new API-based workflow that replaces the SSH/SCP approach with opentrons.simulate and opentrons.execute.

## Key Changes:
- ✅ Replaced Jupyter notebook protocol execution with FastAPI server
- ✅ Uses `opentrons.simulate` for protocol validation
- ✅ API endpoints for protocol simulation and execution
- ✅ Cloud-hosted via Railway (configurable)
- ✅ Maintains compatibility with existing optimization workflow

In [ ]:
# Import required libraries
import requests
import json
import pandas as pd

# Configure API URL - change this to your Railway deployment URL
API_BASE_URL = "http://localhost:8000"  # Local development
# API_BASE_URL = "https://your-railway-app.railway.app"  # Production

## 1. Check API Health

First, verify that the API server is running and healthy.

In [ ]:
# Check API health
response = requests.get(f"{API_BASE_URL}/health")
print(f"API Status: {response.status_code}")
print(f"Response: {response.json()}")

## 2. Prepare Experimental Data

Create experimental data in the same format as before.

In [ ]:
# Example experimental data (same format as before)
experimental_data = [
    {
        'trial_index': '0',
        'drug_name': 'IBP',
        'drug': '120.0',
        's1': '300.0',
        's2': '0.0',
        's3': '0.0',
        's4': '0.0',
        's5': '0.0',
        's6': '0.0',
        's7': '0.0',
        's8': '0.0',
        'dmso': '60.0',
        'water': '900.0'
    },
    {
        'trial_index': '1',
        'drug_name': 'LOV',
        'drug': '180.0',
        's1': '0.0',
        's2': '400.0',
        's3': '0.0',
        's4': '0.0',
        's5': '0.0',
        's6': '0.0',
        's7': '0.0',
        's8': '0.0',
        'dmso': '0.0',
        'water': '800.0'
    }
]

print(f"Prepared {len(experimental_data)} experimental conditions")
print(json.dumps(experimental_data[0], indent=2))

## 3. Simulate Protocol

Use the API to simulate the protocol with opentrons.simulate before running on real hardware.

In [ ]:
# Simulate protocol using the API
simulation_payload = {
    "data": experimental_data,
    "iteration": 1,
    "plate_well": "A1",
    "deepplate_well": "A1"
}

print("Simulating protocol...")
response = requests.post(f"{API_BASE_URL}/simulate", json=simulation_payload, timeout=30)

if response.status_code == 200:
    simulation_result = response.json()
    
    if simulation_result['success']:
        print("✅ Protocol simulation successful!")
        print(f"Protocol text length: {len(simulation_result['protocol_text'])} characters")
        print(f"Simulation log: {simulation_result['run_log']}")
        
        # Save the protocol for inspection
        with open('simulated_protocol.py', 'w') as f:
            f.write(simulation_result['protocol_text'])
        print("Protocol saved to 'simulated_protocol.py'")
    else:
        print(f"❌ Simulation failed: {simulation_result.get('error')}")
else:
    print(f"❌ API request failed: {response.status_code}")
    print(response.text)

## 4. Execute Protocol (When Connected to Robot)

Once the simulation is successful, execute the protocol on real hardware.

In [ ]:
# Execute protocol on robot (requires actual hardware connection)
if 'simulation_result' in locals() and simulation_result['success']:
    execution_payload = {
        "protocol_text": simulation_result['protocol_text'],
        "run_id": "demo_iteration_1"
    }
    
    print("Executing protocol on robot...")
    # Note: This will only work when connected to actual Opentrons hardware
    response = requests.post(f"{API_BASE_URL}/execute", json=execution_payload, timeout=60)
    
    if response.status_code == 200:
        execution_result = response.json()
        
        if execution_result['success']:
            print(f"✅ Protocol executed successfully! Run ID: {execution_result['run_id']}")
            print(f"Status: {execution_result['status']}")
        else:
            print(f"❌ Execution failed: {execution_result.get('error')}")
    else:
        print(f"❌ Execution request failed: {response.status_code}")
        print(response.text)
else:
    print("⚠️ Skipping execution - simulation was not successful or not run")

## 5. Integration with Existing Optimization Workflow

The new API can be integrated into the existing Ax optimization workflow.

In [ ]:
# Example of how to integrate with the optimization workflow
# This replaces the SSH/SCP file upload mechanism

def run_experiment_iteration_api(iteration_data, iteration_number):
    """
    Run an experiment iteration using the API instead of SSH/SCP
    """
    # 1. Simulate the protocol first
    simulation_payload = {
        "data": iteration_data,
        "iteration": iteration_number,
        "plate_well": "H3",
        "deepplate_well": "H3"
    }
    
    sim_response = requests.post(f"{API_BASE_URL}/simulate", json=simulation_payload)
    
    if sim_response.status_code != 200:
        raise Exception(f"Simulation failed: {sim_response.text}")
    
    sim_result = sim_response.json()
    if not sim_result['success']:
        raise Exception(f"Simulation failed: {sim_result.get('error')}")
    
    print(f"✅ Iteration {iteration_number} simulation successful")
    
    # 2. Execute on robot if simulation passes
    execution_payload = {
        "protocol_text": sim_result['protocol_text'],
        "run_id": f"iteration_{iteration_number}"
    }
    
    exec_response = requests.post(f"{API_BASE_URL}/execute", json=execution_payload)
    
    if exec_response.status_code != 200:
        raise Exception(f"Execution failed: {exec_response.text}")
    
    exec_result = exec_response.json()
    if not exec_result['success']:
        raise Exception(f"Execution failed: {exec_result.get('error')}")
    
    print(f"✅ Iteration {iteration_number} executed successfully")
    return exec_result

# Example usage
try:
    result = run_experiment_iteration_api(experimental_data, 1)
    print(f"Experiment completed: {result}")
except Exception as e:
    print(f"Experiment failed: {e}")

## Summary

This new API-based workflow provides several advantages:

1. **Protocol Validation**: Uses `opentrons.simulate` to validate protocols before execution
2. **Cloud Deployment**: Can be deployed on Railway or other cloud platforms
3. **API-Based**: RESTful API for integration with various clients
4. **Maintains Compatibility**: Works with existing optimization and data processing workflows
5. **Error Handling**: Better error reporting and debugging capabilities
6. **Scalability**: Can handle multiple concurrent requests

### Migration from Jupyter Notebooks:

- **Before**: Jupyter notebook → SSH/SCP file upload → Manual robot operation
- **After**: API client → Protocol simulation → Automated robot execution

The API server replaces the need for manual file uploads and provides a more robust, scalable solution for protocol management.